# 15 — Full Evaluation Suite on Colab (GPU)

Runs all evaluation scripts on a GPU runtime for rigorous numbers:

1. **Model comparison** — encoders + SVM + student on full 63k test; decoders at 5k
2. **Fairness** — all models (incl. D2, D1-ft) with perturbation-augmented probes
3. **Encoder–decoder agreement** — toxic-bert vs D1-ft (the better decoder)
4. **Robustness** — bidirectional: evasion (toxic) + FP induction (non-toxic), n=1000

## Colab steps
1. Runtime → Change runtime type → **GPU**
2. Upload/clone repo so `src/`, `scripts/`, `data/raw/jigsaw/`, and `models/` are available
3. Run all cells (~2 hours total)
4. Download `data/processed/*.json` and `report/figures/*.png` back to laptop

**Estimated runtimes (T4 GPU):**
- Encoder + SVM + student full test: ~60 min
- Decoder (5k): ~30 min
- Fairness + perturbation probes: ~10 min
- Agreement (D1-ft): ~15 min
- Robustness bidirectional: ~20 min

In [ ]:
# Colab: uncomment if needed
# !pip install -q transformers datasets scikit-learn pandas pyarrow accelerate

import os, sys
from pathlib import Path

ROOT = Path.cwd()
if (ROOT / "src").exists():
    pass
elif (ROOT / "DeepProjectHate" / "src").exists():
    ROOT = ROOT / "DeepProjectHate"
elif (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("ROOT", ROOT.resolve())

In [ ]:
import torch
from device import get_device

dev = get_device()
print(f"Device: {dev}")
if dev.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — runs will be very slow!")

In [ ]:
from splits import split_counts
print("Dataset split sizes:")
print(split_counts().to_string(index=False))

## 1. Model comparison — encoders + SVM + student (full 63k test)

In [ ]:
!python scripts/run_model_comparison.py --models toxic-bert tfidf-svm distilbert-student

## 2. Model comparison — decoders at eval-n=5000

In [ ]:
!python scripts/run_model_comparison.py --models distilgpt2-d1 distilgpt2-d1-ft --eval-n 5000

In [ ]:
import json
from paths import PROCESSED_DIR

with open(PROCESSED_DIR / "model_comparison.json") as f:
    mc = json.load(f)

print(f"Models evaluated: {list(mc['models'].keys())}")
print(f"Eval sizes: {mc['eval_n']}")
print()
for key, block in mc["models"].items():
    t = block["splits"].get("test", {})
    print(f"{key:25s}  params={block['n_params']:>10,}  "
          f"F1={t.get('f1', 0):.3f}  PR-AUC={t.get('pr_auc', 0):.3f}  "
          f"ROC-AUC={t.get('roc_auc', 0):.3f}  {t.get('ms_per_comment', 0):.1f} ms/comment")

## 3. Fairness — all models + D2 + D1-ft + perturbation probes

In [ ]:
!python scripts/run_fairness.py \
    --models toxic-bert tfidf-svm distilbert-student distilgpt2-d2 distilgpt2-d1-ft \
    --perturb

In [ ]:
with open(PROCESSED_DIR / "fairness_results.json") as f:
    fr = json.load(f)

print("Fairness summary (unperturbed):")
for key, block in fr["models"].items():
    print(f"  {key:25s}  identity_FPR={block['overall_fpr']:.3f}  "
          f"control_FPR={block['control_fpr']:.3f}")
    if "perturbed" in block:
        print(f"    perturbed FPR by perturbation:")
        for pname, pdata in block["perturbed"].items():
            print(f"      {pname:12s}  identity={pdata['overall_fpr']:.3f}  "
                  f"control={pdata['control_fpr']:.3f}")

## 4. Encoder–decoder agreement (toxic-bert vs D1-ft)

In [ ]:
!python scripts/run_agreement.py --n 500

In [ ]:
with open(PROCESSED_DIR / "agreement_results.json") as f:
    ag = json.load(f)

print(f"n={ag['n']}  overall agreement={ag['overall_enc_dec_agreement']:.3f}")
print(f"Disagreements on toxic: {ag['n_disagree_toxic']}/{ag['n']}")
print()
print("vs gold:")
for row in ag["vs_gold"]:
    print(f"  {row['label']:15s}  encoder={row['encoder_vs_gold']:.3f}  "
          f"decoder={row['decoder_vs_gold']:.3f}")

## 5. Robustness — bidirectional (evasion + FP induction), n=1000

In [ ]:
!python scripts/run_robustness.py --models toxic-bert tfidf-svm distilbert-student --n 1000

In [ ]:
with open(PROCESSED_DIR / "robustness_results.json") as f:
    rb = json.load(f)

print(f"Toxic probe: {rb['n_toxic']} comments  |  Non-toxic probe: {rb['n_nontoxic']} comments")
print()
for key, block in rb["models"].items():
    tp = block["toxic_probe"]
    np_block = block["nontoxic_probe"]
    print(f"=== {key} ===")
    print(f"  Toxic probe: baseline detection={tp['baseline_detection_rate']:.3f}")
    for pname, pd_ in tp["perturbations"].items():
        print(f"    {pname:12s}  evasion={pd_['evasion_rate']:.3f}")
    print(f"  Non-toxic probe: baseline FP rate={np_block['baseline_fp_rate']:.3f}")
    for pname, pd_ in np_block["perturbations"].items():
        print(f"    {pname:12s}  FP-induction={pd_['fp_induction_rate']:.3f}")
    print()

## 6. Summary — download these files

Copy back to your laptop:
- `data/processed/model_comparison.json`
- `data/processed/fairness_results.json`
- `data/processed/agreement_results.json`
- `data/processed/robustness_results.json`
- `report/figures/*.png`

In [ ]:
import shutil
from pathlib import Path

out = Path("colab_results")
out.mkdir(exist_ok=True)

for f in (PROCESSED_DIR).glob("*.json"):
    shutil.copy2(f, out / f.name)
    print(f"Copied {f.name}")

fig_dir = Path("report/figures")
if fig_dir.exists():
    for f in fig_dir.glob("*.png"):
        shutil.copy2(f, out / f.name)
        print(f"Copied {f.name}")

print(f"\nAll results in {out.resolve()} — download this folder.")